<a href="https://colab.research.google.com/github/harishkulkarni10/ai-research-assistant-rag/blob/dev/notebooks/01_ingestion/arxiv_ingestion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install datasets pandas pyarrow

In [2]:
from datasets import load_dataset
import pandas as pd
import numpy as np

In [3]:
from datasets import load_dataset
import pandas as pd
from pathlib import Path

In [4]:
load_dataset

<function datasets.load.load_dataset(path: str, name: Optional[str] = None, data_dir: Optional[str] = None, data_files: Union[str, collections.abc.Sequence[str], collections.abc.Mapping[str, Union[str, collections.abc.Sequence[str]]], NoneType] = None, split: Union[str, datasets.splits.Split, list[str], list[datasets.splits.Split], NoneType] = None, cache_dir: Optional[str] = None, features: Optional[datasets.features.features.Features] = None, download_config: Optional[datasets.download.download_config.DownloadConfig] = None, download_mode: Union[datasets.download.download_manager.DownloadMode, str, NoneType] = None, verification_mode: Union[datasets.utils.info_utils.VerificationMode, str, NoneType] = None, keep_in_memory: Optional[bool] = None, save_infos: bool = False, revision: Union[str, datasets.utils.version.Version, NoneType] = None, token: Union[bool, str, NoneType] = None, streaming: bool = False, num_proc: Optional[int] = None, storage_options: Optional[dict] = None, **config_kwargs) -> Union[datasets.dataset_dict.DatasetDict, datasets.arrow_dataset.Dataset, datasets.dataset_dict.IterableDatasetDict, datasets.iterable_dataset.IterableDataset]>

## Step 1 & 2: Load the dataset

Dataset: ccdv/arxiv-summarization  
Contains ~215k arXiv papers with clean full article text + abstract. No PDF processing needed.

# Loading the arXiv dataset

In [11]:
# Load the dataset (it has train/validation/test splits — we'll use train for corpus)
ds = load_dataset("ccdv/arxiv-summarization", split="train")

print(f"Dataset loaded!")
print(f"Total papers: {len(ds):,}")
print("\nSchema:")
print(ds.features)
print("\nSample entry keys:", list(ds[0].keys()))

README.md: 0.00B [00:00, ?B/s]

section/train-00000-of-00015.parquet:   0%|          | 0.00/230M [00:00<?, ?B/s]

section/train-00001-of-00015.parquet:   0%|          | 0.00/228M [00:00<?, ?B/s]

section/train-00002-of-00015.parquet:   0%|          | 0.00/228M [00:00<?, ?B/s]

section/train-00003-of-00015.parquet:   0%|          | 0.00/227M [00:00<?, ?B/s]

...

section/test-00000-of-00001.parquet:   0%|          | 0.00/105M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/203037 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/6436 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/6440 [00:00<?, ? examples/s]

Dataset loaded!
Total papers: 203,037

Schema:
{'article': Value('string'), 'abstract': Value('string')}

Sample entry keys: ['article', 'abstract']


## Step 3: Filter to AI/ML papers

Simple keyword filtering on title + abstract (lowercase).
Keywords chosen to capture core AI/ML topics without too much noise.

In [14]:
# Define AI/ML keywords (we can expand this as the project goes ahead)

ai_ml_keywords = [
    "machine learning", "deep learning", "neural network", "transformer",
    "large language model", "llm", "gpt", "bert", "artificial intelligence",
    "computer vision", "natural language processing", "nlp", "reinforcement learning",
    "generative ai", "diffusion model", "rag", "retrieval augmented"
]

# Combining title and abstract for searching
def contains_ai_ml(example):
  text = (example['article'] + ' ' + example['abstract']).lower()
  return any(keyword in text for keyword in ai_ml_keywords)

# Filter the dataset
filtered_ds = ds.filter(contains_ai_ml)

print(f"Filtered papers: {len(filtered_ds):,}")
print(f"Reduction: {len(ds) - len(filtered_ds):,} papers removed")

Filter:   0%|          | 0/203037 [00:00<?, ? examples/s]

Filtered papers: 155,119
Reduction: 47,918 papers removed


## Step 4: Limit dataset size (baseline)

We'll take 1,000 papers for fast prototyping.
Shuffled with fixed seed for reproducibility.

In [17]:
np.random.seed(42)
limited_ds = filtered_ds.shuffle(seed=42).select(range(1000))

print(f"Final baseline corpus size: {len(limited_ds):,} papers")

Final baseline corpus size: 1,000 papers


## Step 5: Create clean DataFrame corpus

Columns:
- paper_id: index (temporary unique ID)
- title: not directly available → we'll use first 200 chars of abstract as proxy
- abstract: full abstract
- full_text: article body

In [20]:
df = pd.DataFrame(limited_ds)

# paper_id
df['paper_id'] = range(len(df))

df['title'] = df['abstract'].str.slice(0, 200) + "..."

df = df.rename(columns={'article': 'full_text'})

# Reorder columns
df = df[['paper_id', 'title', 'abstract', 'full_text']]

print("Corpus DataFrame created!")
df.head()

Corpus DataFrame created!


,paper_id,title,abstract,full_text
0,0,we use new observations of very weak civ absor...,we use new observations of very weak civ absor...,the observational result that high - redshift ...
1,1,we present the detection of a @xmath0 black ho...,we present the detection of a @xmath0 black ho...,the questions of how the nuclei of galaxies fo...
2,2,we study harmonic functions on random environm...,we study harmonic functions on random environm...,"since the work of yau @xcite in 1975 , where t..."
3,3,we study pseudoscalar and scalar mesons using ...,we study pseudoscalar and scalar mesons using ...,dyson - schwinger equations ( dses ) are a non...
4,4,we present our recent results on mm - wave co ...,we present our recent results on mm - wave co ...,"after attending this conference , i think that..."


# Save artifact

In [21]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [22]:
drive_path = Path("/content/drive/MyDrive/rag-arxiv-qa/data/processed")
drive_path.mkdir(parents=True, exist_ok=True)

output_path = drive_path / "arxiv_ai_ml_corpus.parquet"

df.to_parquet(output_path, index=False)

print(f"Saved to Google Drive: {output_path}")

Saved to Google Drive: /content/drive/MyDrive/rag-arxiv-qa/data/processed/arxiv_ai_ml_corpus.parquet


# Validation

In [23]:
print(f"Final number of papers: {len(df):,}")

# Average lengths
df['abstract_len'] = df['abstract'].str.len()
df['full_text_len'] = df['full_text'].str.len()

print(f"\nAverage abstract length: {df['abstract_len'].mean():.0f} chars")
print(f"Average full text length: {df['full_text_len'].mean():.0f} chars")

# Sample inspection
print("\nSample 1:")
print("Title proxy:", df.iloc[0]['title'])
print("Abstract preview:", df.iloc[0]['abstract'][:500])
print("Full text preview:", df.iloc[0]['full_text'][:500])

print("\nSample 2:")
print("Title proxy:", df.iloc[1]['title'])
print("Abstract preview:", df.iloc[1]['abstract'][:500])

Final number of papers: 1,000

Average abstract length: 1436 chars
Average full text length: 37781 chars

Sample 1:
Title proxy: we use new observations of very weak civ absorption lines associated with high - redshift ly @xmath0  absorption systems to measure the high - redshift ly @xmath0  line two - point correlation functio...
Abstract preview: we use new observations of very weak civ absorption lines associated with high - redshift ly @xmath0  absorption systems to measure the high - redshift ly @xmath0  line two - point correlation function ( tpcf ) . 
 these very weak civ absorption lines trace small - scale velocity structure that can not be resolved by ly @xmath0absorption lines . 
 we find that ( 1 )  high - redshift ly @xmath0  absorption systems with @xmath1  are strongly clustered in redshift , ( 2 )  previous measurements of 
Full text preview: the observational result that high - redshift ly @xmath0  absorption systems appear not to cluster strongly in redshift ( e.g. sa